In [1]:
!pip install transformers gradio sentencepiece -q
!pip install reportlab
!pip install reportlab gradio transformers sentencepiece
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from transformers.utils import logging
import warnings
warnings.filterwarnings('ignore')
logging.set_verbosity_error()

MODEL_NAME = 't5-small'

print('Loading model...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model     = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
print('Model ready!')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 24.3 MB/s eta 0:00:00
Loading model...


config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  242MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Model ready!


In [2]:
def summarize(text, max_words=60):
    # T5 needs 'summarize: ' at the start
    input_text = 'summarize: ' + text

    # Convert text to numbers
    inputs = tokenizer(
        input_text,
        return_tensors = 'pt',
        max_length     = 512,
        truncation     = True
    )

    # Generate the summary
    output = model.generate(
        **inputs,
        max_new_tokens  = max_words,
        min_length      = 10,
        length_penalty  = 2.0,
        num_beams       = 4,
        early_stopping  = True
    )

    # Convert numbers back to text
    result = tokenizer.decode(output[0], skip_special_tokens=True)
    return result

print('summarize() function ready!')


summarize() function ready!


In [3]:

# Sample text
long_text = """
Artificial Intelligence (AI) is transforming many industries around the world.
It helps computers perform tasks that usually require human intelligence.
Machine learning is a part of AI that allows computers to learn from data.
Deep learning uses neural networks to solve complex problems.
AI is widely used in healthcare, education, banking, transportation, and agriculture.
In healthcare, AI helps doctors detect diseases and analyze medical images.
In education, AI supports personalized learning and automated assessment.
As technology advances, AI is becoming an important part of modern life.
"""

summary = summarize(long_text)

print("ORIGINAL TEXT:")
print(long_text)

print("\nGENERATED SUMMARY:")
print(summary)


ORIGINAL TEXT:

Artificial Intelligence (AI) is transforming many industries around the world.
It helps computers perform tasks that usually require human intelligence.
Machine learning is a part of AI that allows computers to learn from data.
Deep learning uses neural networks to solve complex problems.
AI is widely used in healthcare, education, banking, transportation, and agriculture.
In healthcare, AI helps doctors detect diseases and analyze medical images.
In education, AI supports personalized learning and automated assessment.
As technology advances, AI is becoming an important part of modern life.


GENERATED SUMMARY:
machine learning is a part of AI that allows computers to learn from data. AI is widely used in healthcare, education, banking, transportation, and agriculture.


In [4]:

articles = [
    {
        "title": "Solar Energy",
        "text": """
        Solar energy is a renewable source of energy obtained from sunlight.
        Solar panels convert sunlight into electricity for homes and businesses.
        Solar power reduces dependence on fossil fuels and helps lower carbon emissions.
        Although installation can be expensive, solar panels can reduce electricity bills
        over time. Solar energy is becoming increasingly popular worldwide.
        """
    },
    {
        "title": "Electric Vehicles",
        "text": """
        Electric vehicles use electric motors powered by rechargeable batteries.
        They produce no exhaust emissions while driving and can reduce air pollution.
        Many governments encourage electric vehicle adoption through incentives.
        Improvements in battery technology are increasing driving range and reducing
        charging time. Charging infrastructure is also expanding in many countries.
        """
    }
]

for article in articles:
    print("=" * 50)
    print("TITLE:", article["title"])
    print("\nSUMMARY:")
    print(summarize(article["text"]))
    print()


TITLE: Solar Energy

SUMMARY:
solar panels convert sunlight into electricity for homes and businesses. solar power reduces dependence on fossil fuels and helps lower carbon emissions.

TITLE: Electric Vehicles

SUMMARY:
electric motors powered by rechargeable batteries produce no exhaust emissions while driving. many governments encourage electric vehicle adoption through incentives.



In [5]:

import gradio as gr
import tempfile
import html
import os


from reportlab.lib.pagesizes import A4
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer
)
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.colors import HexColor
from reportlab.lib.units import inch


# ============================================================
# SUMMARIZATION + PDF DOWNLOAD
# Requires your existing summarize(text, max_words) function.
# ============================================================

def summarize_app(text, max_words):
    if not text or not text.strip():
        return (
            "Paste your article, report, or long text here.",
            "Word Count: —  |  Compression: —  |  Readability: —",
            None
        )

    if len(text.split()) < 20:
        return (
            "Please enter at least 20 words to generate a summary.",
            "Word Count: —  |  Compression: —  |  Readability: —",
            None
        )

    pdf_path = None

    try:
        max_words = int(max_words)
        result = summarize(text.strip(), max_words=max_words)

        original_words = len(text.split())
        summary_words = len(result.split())

        compression = (
            round(original_words / summary_words, 1)
            if summary_words else 0
        )

        stats = (
            f"Word Count: {summary_words}  |  "
            f"Compression: {compression}:1  |  "
            f"Readability: —"
        )

        # Generate a real PDF file
        with tempfile.NamedTemporaryFile(
            suffix=".pdf",
            prefix="briefly_ai_",
            delete=False
        ) as temp:
            pdf_path = temp.name

        styles = getSampleStyleSheet()

        title_style = ParagraphStyle(
            "BrieflyTitle",
            parent=styles["Title"],
            textColor=HexColor("#24364B"),
            fontSize=22,
            leading=28,
            spaceAfter=18
        )

        body_style = ParagraphStyle(
            "BrieflyBody",
            parent=styles["BodyText"],
            textColor=HexColor("#303B49"),
            fontSize=11,
            leading=17
        )

        doc = SimpleDocTemplate(
            pdf_path,
            pagesize=A4,
            rightMargin=0.75 * inch,
            leftMargin=0.75 * inch,
            topMargin=0.75 * inch,
            bottomMargin=0.75 * inch
        )

        story = [
            Paragraph("Briefly.AI", title_style),
            Paragraph(
                "Intelligent Text Summary",
                styles["Heading2"]
            ),
            Spacer(1, 12),
            Paragraph(
                html.escape(result).replace("\n", "<br/>"),
                body_style
            ),
            Spacer(1, 24),
            Paragraph(html.escape(stats), styles["Italic"])
        ]

        doc.build(story)

        return result, stats, pdf_path

    except Exception as error:
        if pdf_path and os.path.exists(pdf_path):
            os.remove(pdf_path)

        return (
            f"Unable to generate summary: {error}",
            "Please try again.",
            None
        )


# ============================================================
# CUSTOM CSS — BRIEFLY.AI
# Footer text and decorative watermark removed.
# ============================================================

custom_css = """
html, body {
    margin: 0 !important;
    padding: 0 !important;
    background: #202D3C !important;
}

.gradio-container {
    max-width: 100% !important;
    min-height: 100vh !important;
    margin: 0 !important;
    padding: 0 !important;
    background: #202D3C !important;
    color: #F6EBDD !important;
    font-family: Inter, "Segoe UI", Arial, sans-serif !important;
}

footer {
    display: none !important;
}

/* Main page */
#briefly-app {
    position: relative;
    isolation: isolate;
    overflow: hidden;
    min-height: 100vh;
    box-sizing: border-box;
    padding: 0 24px 36px;
    color: #F6EBDD;

    background:
        radial-gradient(
            ellipse at 52% 43%,
            rgba(128,145,164,.15),
            transparent 42%
        ),
        linear-gradient(
            155deg,
            #202D3C 0%,
            #253347 52%,
            #365A8D 100%
        );
}

/* Decorative blue waves — retained */
#briefly-app:before {
    content: "";
    position: absolute;
    z-index: -1;
    pointer-events: none;
    left: -9%;
    right: -9%;
    bottom: 10%;
    height: 290px;

    background: linear-gradient(
        180deg,
        rgba(89,119,156,.12),
        rgba(51,85,132,.76)
    );

    border-top: 1px solid rgba(215,224,233,.28);
    border-radius: 49% 54% 0 0 / 27% 38% 0 0;
    transform: rotate(-4deg);

    box-shadow:
        0 -18px 0 -17px rgba(225,229,230,.28),
        0 -43px 0 -42px rgba(122,156,193,.35);
}

/* Star watermark removed */

/* Header */
.brand {
    padding-top: 22px;
    text-align: center;
    font-size: 29px;
    font-weight: 750;
    letter-spacing: -.8px;
    color: #F7EBDD;
    text-shadow: 0 0 20px rgba(255,234,198,.16);
}

.hero-title {
    margin: 38px 0 9px;
    text-align: center;
    color: #F8EEDF;
    font-size: clamp(32px,4vw,60px);
    line-height: 1.12;
    font-weight: 750;
    letter-spacing: -1.5px;
}

.hero-subtitle {
    max-width: 1000px;
    margin: 14px auto 38px;
    text-align: center;
    color: #BFC5CE;
    font-size: clamp(14px,1.45vw,21px);
    line-height: 1.55;
}

/* Two-column workspace */
#workspace {
    max-width: 1300px;
    width: 100%;
    margin: 0 auto;
    align-items: stretch !important;
    gap: 26px !important;
}

.glass-card {
    box-sizing: border-box;
    min-width: 0;
    min-height: 365px;
    padding: 22px !important;

    border-radius: 19px !important;
    border: 1px solid rgba(202,215,228,.22) !important;
    background: rgba(54,68,86,.60) !important;

    backdrop-filter: blur(18px);
    -webkit-backdrop-filter: blur(18px);

    box-shadow:
        0 15px 40px rgba(9,19,33,.12),
        inset 0 1px 0 rgba(255,255,255,.035) !important;
}

/* Input card */
#input-card {
    border: 2px solid rgba(249,239,218,.92) !important;

    background:
        radial-gradient(
            ellipse at 48% 0%,
            rgba(240,223,189,.11),
            transparent 65%
        ),
        rgba(57,70,87,.67) !important;

    box-shadow:
        0 0 20px rgba(255,229,176,.13),
        0 0 2px rgba(255,239,208,.45) !important;
}

.card-heading {
    color: #F7EBDD;
    font-size: 21px;
    font-weight: 500;
    margin: 0 0 14px;
}

/* Cream text input */
#briefly-app textarea {
    color: #354052 !important;
    background: #F8F0DF !important;
    border: 1px solid rgba(232,218,191,.9) !important;
    border-radius: 9px !important;
    box-shadow: none !important;
    font-family: Inter, "Segoe UI", Arial, sans-serif !important;
    font-size: 16px !important;
    line-height: 1.55 !important;
    padding: 13px !important;
}

#briefly-app textarea::placeholder {
    color: #77777A !important;
    opacity: 1 !important;
}

/* Summary length heading */
#length-heading {
    margin-top: 19px;
    margin-bottom: 12px;
}

/* Gold slider — no number input boxes */
#length-slider {
    width: 100% !important;
    min-width: 0 !important;
    margin: 0 !important;
}

#length-slider input[type="range"] {
    accent-color: #D7BC82 !important;
}

#length-slider input[type="range"]::-webkit-slider-runnable-track {
    background: linear-gradient(
        90deg,
        #B99458,
        #F4DFAD,
        #6F7A88
    ) !important;
}

#length-slider input[type="range"]::-webkit-slider-thumb {
    background: radial-gradient(
        circle at 35% 28%,
        #FFF1C7,
        #D5B779 52%,
        #80643B
    ) !important;
    border: 1px solid #F7E3B0 !important;
    box-shadow: 0 0 9px rgba(243,213,151,.25);
}

/* Concise / Detailed labels */
#length-labels {
    display: flex;
    align-items: center;
    justify-content: space-between;
    gap: 12px;
    margin-top: 8px;
    color: #F1EBDD;
    font-size: 14px;
}

/* Summary output */
#summary-output,
#summary-output textarea {
    color: #F7F7F7 !important;
    background: transparent !important;
    border: none !important;
    box-shadow: none !important;
}

#summary-output textarea {
    min-height: 190px !important;
    padding: 0 !important;
    font-size: 19px !important;
    line-height: 1.48 !important;
}

/* Statistics */
#stats {
    box-sizing: border-box;
    width: 100%;
    padding: 11px 12px;
    margin-top: 10px;
    border-radius: 9px;
    color: #F5F1E9;
    background: rgba(151,172,196,.20);
    text-align: center;
    font-size: 15px;
    line-height: 1.5;
    overflow-wrap: anywhere;
}

/* Blue PDF button */
#download-btn {
    width: 100%;
    margin-top: 14px;
}

#download-btn button {
    width: 100% !important;
    min-height: 48px !important;
    border: 1px solid rgba(255,255,255,.13) !important;
    border-radius: 10px !important;

    background: linear-gradient(
        110deg,
        #3478F6,
        #2E72ED
    ) !important;

    color: #FFFFFF !important;
    font-size: 20px !important;
    font-weight: 500 !important;
    box-shadow: 0 5px 17px rgba(32,93,208,.22) !important;
    transition: all .25s ease !important;
}

#download-btn button:hover {
    background: linear-gradient(
        110deg,
        #4A8AFF,
        #3478F6
    ) !important;
    transform: translateY(-1px);
}

/* Cream Summarize button */
#summarize-btn {
    width: 100%;
    max-width: 290px;
    margin: 28px auto 0 !important;
}

#summarize-btn button {
    width: 100% !important;
    min-height: 73px !important;
    border-radius: 50px !important;
    border: 1px solid rgba(255,246,225,.96) !important;

    background: linear-gradient(
        110deg,
        #FFF8E8 0%,
        #E8DFCB 45%,
        #FFF7E4 100%
    ) !important;

    color: #26384D !important;
    font-size: 25px !important;
    font-weight: 550 !important;
    box-shadow: 0 0 23px rgba(255,236,194,.27) !important;
    transition: all .25s ease !important;
}

#summarize-btn button:hover {
    background: linear-gradient(
        110deg,
        #FFFFFF,
        #F0E7D5,
        #FFFFFF
    ) !important;
    transform: translateY(-2px);
}

/* Hide any leftover footer if present */
.footer-note {
    display: none !important;
}

/* Responsive layout */
@media (max-width: 760px) {
    #briefly-app {
        padding: 0 16px 24px;
    }

    .brand {
        padding-top: 20px;
        font-size: 25px;
    }

    .hero-title {
        margin-top: 27px;
        font-size: 34px;
    }

    .hero-subtitle {
        margin-bottom: 25px;
        font-size: 14px;
    }

    #workspace {
        gap: 16px !important;
    }

    .glass-card {
        min-height: auto;
        padding: 17px !important;
    }

    #summary-output textarea {
        min-height: 150px !important;
        font-size: 16px !important;
    }

    #summarize-btn {
        margin-top: 20px !important;
    }

    #summarize-btn button {
        min-height: 58px !important;
        font-size: 20px !important;
    }
}
"""


# ============================================================
# BUILD THE GRADIO PAGE
# ============================================================

with gr.Blocks(
    css=custom_css,
    title="Briefly.AI — Less Noise. More Clarity.",
    theme=gr.themes.Base(
        primary_hue="blue",
        secondary_hue="slate",
        neutral_hue="slate"
    )
) as demo:

    with gr.Column(elem_id="briefly-app"):

        # Header — no Gemini branding
        gr.HTML("""
            <div class="brand">✦ Briefly.AI</div>

            <div class="hero-title">
                Less Noise. More Clarity.
            </div>

            <div class="hero-subtitle">
                Instantly summarize any text, from academic papers to
                long articles, with human-level precision.
            </div>
        """)

        with gr.Row(elem_id="workspace"):

            # LEFT CARD
            with gr.Column(
                scale=1,
                elem_classes=["glass-card"],
                elem_id="input-card"
            ):
                gr.HTML(
                    '<div class="card-heading">Your Text</div>'
                )

                text_input = gr.Textbox(
                    placeholder=(
                        "Paste your article, report, or long text "
                        "here for an intelligent summary..."
                    ),
                    lines=5,
                    max_lines=10,
                    show_label=False,
                    container=False
                )

                gr.HTML("""
                    <div id="length-heading" class="card-heading">
                        Summary Length
                    </div>
                """)

                # Single slider; no extra number boxes
                length_slider = gr.Slider(
                    minimum=20,
                    maximum=100,
                    value=50,
                    step=10,
                    show_label=False,
                    container=False,
                    elem_id="length-slider"
                )

                gr.HTML("""
                    <div id="length-labels">
                        <span>Concise</span>
                        <span>Detailed</span>
                    </div>
                """)

            # RIGHT CARD
            with gr.Column(
                scale=1,
                elem_classes=["glass-card"]
            ):
                gr.HTML("""
                    <div class="card-heading">
                        Intelligent Summary
                    </div>
                """)

                summary_output = gr.Textbox(
                    value=(
                        "Your generated summary will appear here. "
                        "Paste your text on the left and click "
                        "Summarize to see your results."
                    ),
                    show_label=False,
                    lines=7,
                    max_lines=10,
                    interactive=False,
                    container=False,
                    elem_id="summary-output"
                )

                stats_output = gr.HTML("""
                    <div id="stats">
                        Word Count: — &nbsp; | &nbsp;
                        Compression: — &nbsp; | &nbsp;
                        Readability: —
                    </div>
                """)

                download_button = gr.DownloadButton(
                    label="Download PDF",
                    value=None,
                    variant="primary",
                    elem_id="download-btn"
                )

        summarize_button = gr.Button(
            "✦  Summarize",
            variant="primary",
            elem_id="summarize-btn"
        )

        # Footer HTML removed completely.
        # Decorative star watermark CSS removed completely.

    # Send the slider value directly to the model
    summarize_button.click(
        fn=summarize_app,
        inputs=[text_input, length_slider],
        outputs=[
            summary_output,
            stats_output,
            download_button
        ]
    )


demo.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f3db73477ee4a16465.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
